In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('toy_data.csv', sep=r'\s*,\s*', index_col=False)
df.head(5)

In [ ]:
recepient_accs =set(df['from'])
recepient_accs

In [ ]:
# User accounts send money. Income accounts are foreign accounts that send money to user accounts
user_accts = set(['Checking', 'Chase_CC'])
income_accts = recepient_accs.difference(user_accts) # remove those as they aren't actually showing income
income_accts

In [ ]:
income_txs = df[df['to'].isin(user_accts)]
sum(income_txs['amount'])

In [ ]:
# Now we want to see how much expenses are
#	 =		 is sent from a user and is not a transfer to a user
user_txs = df[df['from'].isin(user_accts) & ~df['to'].isin(user_accts)]
total_expenses = sum(user_txs['amount'])

In [ ]:
# Let's see how much each vendor is
vendor_balance_df = user_txs.groupby('to').agg({'amount': 'sum'})
vendor_balance_df

In [ ]:
# This tells us how much we have to allocate to each
# Now we have virtual accounts
virtual_accs = set([
	'Insurance',
	'Rent',
	'Tuition',
	'Lease',
	'Free Cash',
	'Groceries'
])
vacc2vendor: dict[str, list[str]] = dict(((vacc,[]) for vacc in virtual_accs))
vacc2vendor['Insurance'] = ['Geico']
vacc2vendor['Rent'] = ['Houston']
vacc2vendor['Tuition'] = ['IIT']
vacc2vendor['Lease'] = ['Lease']
vacc2vendor['Groceries'] = ['Target','Trader Joes']
vacc2vendor['Free Cash'] = ['Nails R Us']
vendor2vacc: dict[str, str] = dict()
# Now we want to run the mapping in the reverse
for vacc, vendors in vacc2vendor.items():
	for vendor in vendors:
		vendor2vacc[vendor] = vacc
		

vendors = set(vendor_balance_df.index)
vendors

In [ ]:
vendor_balance_df.loc['Lease']['amount']

# Now we want to convert our income transactions into virtual income transactions

In [ ]:
vendor_balance = dict( (vendor, 0 ) for vendor in [ *vendors, 'Free Cash' ])

vendor_balance

def get_remaining_balance(vendor):
        return vendor_balance_df.loc[vendor]['amount'] - vendor_balance[vendor]
def get_next_vendor_to_fill():
        return sorted(vendors, key=get_remaining_balance)[-1]

In [ ]:
def calculate_contribution(pay, remaining_balance):
        return min(pay, remaining_balance)

income_alloc_txs = []
for idx, paycheck in income_txs.iterrows():
	txid, frm, to, pay = paycheck
	vendor = get_next_vendor_to_fill()
	remaining_balance = get_remaining_balance(vendor)

	while (contribution := calculate_contribution( pay, remaining_balance )) > 0:
		pay -= contribution
		vendor_balance[vendor] += contribution
		income_alloc_txs.append((txid, frm, vendor2vacc[vendor], contribution))
		
		vendor = get_next_vendor_to_fill()
		remaining_balance = get_remaining_balance(vendor)
	
	if pay > 0:
		vendor_balance['Free Cash'] += pay
		income_alloc_txs.append((txid, frm, 'Free Cash', pay))
vendor_balance

In [ ]:
vallocs: pd.DataFrame = pd.DataFrame(income_alloc_txs,columns=['txid','from','to','amount'])
vallocs.index.name = 'vtxid'
vallocs.to_csv('virtual_income.csv')
vallocs.head(5)

# Now we want to calculate our equivalent virtual expense transactions

In [ ]:
expenses = df[~df['to'].isin(user_accts)]
def get_vacc_from_vendor(vendor):
        return vendor2vacc[vendor]
expenses['VACC'] = expenses['to'].apply(get_vacc_from_vendor)

expenses

In [ ]:
virtual_expenses = expenses
virtual_expenses['from'] = virtual_expenses['VACC']
virtual_expenses.drop(columns=[ 'VACC'], inplace=True)
# make sure the virtual indexes don't overlap
virtual_expenses.index += vallocs.index[-1]
virtual_expenses.index.name = 'vtxid'
virtual_expenses

In [ ]:
virtual_expenses.to_csv('virtual_expenses.csv')